# EasyRAG Evaluation Basics

## What you will do

- run both retrieval and answer evaluation on one tiny corpus
- compare what each report can and cannot tell you
- map evaluation outputs back to the pipeline stages they diagnose

## Why this matters

Evaluation is easier to learn when the retrieval report and the answer report sit side by side. That makes the stage boundary visible.


## Source code anchors

- `easyrag.rag.evaluation.evaluate_retrieval`
- `easyrag.rag.evaluation.evaluate_answers`
- `easyrag.rag.types.EvalCase`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.evaluation import evaluate_answers, evaluate_retrieval


## Deterministic path

We use one tiny workspace and a small case set so you can read both reports directly.


In [ ]:
eval_tmp = tempfile.TemporaryDirectory()
eval_rag = EasyRAG(
    working_dir=eval_tmp.name,
    workspace="evaluation-basics-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(eval_rag.initialize_storages())
run_sync(
    eval_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses query rewrite for grounded retrieval.\n",
            "# Policy\nCitation-aware answers preserve traceability.\n",
        ],
        ids=["doc::retrieval", "doc::policy"],
        file_paths=["docs/retrieval.md", "docs/policy.md"],
    )
)
cases = [
    EvalCase(question="What uses query rewrite?", expected_document_ids=("doc::retrieval",), expected_to_abstain=False),
    EvalCase(question="When is the company retreat?", expected_document_ids=(), expected_to_abstain=True),
]
retrieval_report = run_sync(evaluate_retrieval(eval_rag, cases, QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3)))
answer_report = run_sync(evaluate_answers(eval_rag, cases, QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3), AnswerParam()))
_print_json({"retrieval_metrics": retrieval_report["metrics"], "answer_metrics": answer_report["metrics"]})


## Output inspection

Retrieval evaluation tells you whether relevant evidence surfaced. Answer evaluation tells you whether the answer stayed grounded and abstained correctly when needed.


In [ ]:
_print_json({"retrieval_cases": retrieval_report["cases"], "answer_cases": answer_report["cases"]})


## Provider-backed path

The optional path keeps the same report structure when providers are configured, which is useful because the evaluation API should stay stable even when the backend changes.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-evaluation-basics-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(provider_rag.ainsert("# Retrieval\nGrounded retrieval keeps answers inspectable.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
        provider_report = run_sync(evaluate_retrieval(provider_rag, [EvalCase(question="What keeps answers inspectable?", expected_document_ids=("doc::retrieval",))], QueryParam(mode="hybrid", chunk_top_k=2)))
        _print_json(provider_report["metrics"])
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

Evaluation becomes clearer once you separate the questions. Retrieval asks whether the evidence was found. Grounding asks whether the answer stayed inside that evidence. A single score would blur those two failures together.


In [ ]:
run_sync(eval_rag.finalize_storages())
eval_tmp.cleanup()
print("Cleaned up the evaluation-basics workspace.")


## Next steps

- Continue with [06_02_building_a_tiny_eval_set.ipynb](06_02_building_a_tiny_eval_set.ipynb) to make the case-construction step explicit.
- Read [06-evaluation-overview.md](../../docs/06-evaluation-overview.md) for the chapter-level evaluation map.
- Revisit [06_03_retrieval_metrics.ipynb](06_03_retrieval_metrics.ipynb) and [06_04_answer_grounding_and_faithfulness.ipynb](06_04_answer_grounding_and_faithfulness.ipynb) for the deeper stage-specific labs.
